# Leçon Python 1 — Lire et explorer les données avec pandas

| | |
|---|---|
| **Auteure** | Géneviève AMAGBEGNON |
| **Date** | 22/10/2026 |
| **Base** | `assurance_entrainement.db` (SQLite, données fictives) : clients, contrats, sinistres |
| **Notions** | connexion SQLite, `read_sql`, `head`, `shape`, `info`, `describe`, sélection de colonnes, filtres, tri, `value_counts`, `groupby` |
| **Outil** | Jupyter Notebook (Anaconda) |

**Objectif :** explorer et analyser les données d’un portefeuille d’assurance avec pandas (chargement, sélection, filtres, tri et indicateurs par catégorie). Pour chaque opération, l’équivalent SQL est indiqué afin de comparer les deux approches.



## 0. Préparer l'environnement

- `import pandas as pd` : charge la bibliothèque pandas, l'outil n° 1 de l'analyse de données en Python. `pd` est son surnom habituel.
- `import sqlite3` : permet de se connecter à une base SQLite.
- Le chemin `../sql/assurance_entrainement.db` veut dire : remonter d'un dossier (`..`), puis aller dans `sql`. Ce notebook doit donc être rangé dans `data-portfolio/python/`.

In [1]:
import pandas as pd
import sqlite3

# Connexion à la base (le même fichier que dans DB Browser)
connexion = sqlite3.connect("../sql/assurance_entrainement.db")

## 1. Charger les tables dans des DataFrames

Un **DataFrame** est un tableau pandas : des lignes et des colonnes, comme une table SQL ou une feuille Excel.

`pd.read_sql(requête, connexion)` exécute une requête SQL et range le résultat dans un DataFrame. On charge ici chaque table entière.

In [2]:
clients = pd.read_sql("SELECT * FROM clients", connexion)
contrats = pd.read_sql("SELECT * FROM contrats", connexion)
sinistres = pd.read_sql("SELECT * FROM sinistres", connexion)

## 2. Découvrir un tableau

| pandas | Rôle | Équivalent SQL |
|---|---|---|
| `df.head(n)` | les n premières lignes | `SELECT * ... LIMIT n` |
| `df.shape` | (nombre de lignes, nombre de colonnes) | `SELECT COUNT(*)` |
| `df.info()` | colonnes, types, valeurs non nulles | onglet Structure |
| `df.describe()` | statistiques des colonnes numériques | `MIN`, `MAX`, `AVG`… |

In [3]:
# Question : à quoi ressemblent les 5 premiers clients ?
clients.head()

,id_client,nom,prenom,sexe,date_naissance,ville,departement,date_inscription
0,1,Roux,Lea,F,1970-01-10,Paris,75,2019-07-26
1,2,Richard,Ines,F,1987-11-08,Lyon,69,2018-05-11
2,3,Michel,Lea,F,1995-05-04,Argenteuil,95,2024-10-01
3,4,Lefebvre,Nadia,F,1987-08-21,Versailles,78,2020-06-21
4,5,Roux,Jean,M,1964-04-28,Évry,91,2025-10-30


In [4]:
# Question : combien de lignes et de colonnes dans chaque table ?
print("clients   :", clients.shape)
print("contrats  :", contrats.shape)
print("sinistres :", sinistres.shape)
# Résultat : (200, 8), (339, 7), (260, 7)

clients   : (200, 8)
contrats  : (339, 7)
sinistres : (260, 7)


In [5]:
# Question : quelles colonnes, quels types, combien de valeurs renseignées ?
contrats.info()
# À lire : prime_annuelle a 324 valeurs non nulles sur 339 → 15 manquantes (comme en SQL)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 339 entries, 0 to 338
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   id_contrat      339 non-null    int64  
 1   id_client       339 non-null    int64  
 2   type_contrat    339 non-null    object 
 3   date_debut      339 non-null    object 
 4   date_fin        63 non-null     object 
 5   prime_annuelle  324 non-null    float64
 6   statut          339 non-null    object 
dtypes: float64(1), int64(2), object(4)
memory usage: 18.7+ KB


In [6]:
# Question : quelles statistiques sur les primes ?
contrats["prime_annuelle"].describe()
# Résultat : count 324, mean 716,53, min 150,49, max 1 772,22

count     324.000000
mean      716.532377
std       393.124488
min       150.490000
25%       424.437500
50%       610.275000
75%       921.002500
max      1772.220000
Name: prime_annuelle, dtype: float64


Affiche les 3 premières lignes de la table `sinistres`, puis ses dimensions (`shape`).

In [7]:
# Question : quelles sont les 3 premières lignes et les dimensions de la table sinistres ?
display(sinistres.head(3))
print("sinistres :", sinistres.shape)


,id_sinistre,id_contrat,date_sinistre,type_sinistre,montant_declare,montant_rembourse,statut
0,1,3,2021-12-13,Arrêt de travail,4660.06,2936.23,Clos
1,2,4,2026-03-03,Accident,7703.04,0.00,Refusé
2,3,6,2022-05-07,Dégât des eaux,989.01,554.56,Clos


sinistres : (260, 7)


## 3. Choisir ses colonnes

- **Une** colonne : `df["colonne"]` (un seul jeu de crochets).
- **Plusieurs** colonnes : `df[["col1", "col2"]]` (**deux** jeux de crochets : on passe une liste).

Équivalent SQL : `SELECT nom, prenom, ville FROM clients`.

In [8]:
# Question : nom, prénom et ville des clients (5 premiers)
clients[["nom", "prenom", "ville"]].head()

,nom,prenom,ville
0,Roux,Lea,Paris
1,Richard,Ines,Lyon
2,Michel,Lea,Argenteuil
3,Lefebvre,Nadia,Versailles
4,Roux,Jean,Évry



Affiche `id_contrat`, `type_contrat` et `prime_annuelle` des 5 premiers contrats.

In [9]:
# Question : quels sont l'identifiant, le type et la prime des 5 premiers contrats ?
contrats[["id_contrat", "type_contrat", "prime_annuelle"]].head()


,id_contrat,type_contrat,prime_annuelle
0,1,Santé,1662.37
1,2,Santé,571.02
2,3,Prévoyance,618.00
3,4,Auto,457.04
4,5,Habitation,413.87


## 4. Filtrer les lignes

On écrit une **condition entre crochets** : `df[df["colonne"] == valeur]`. Équivalent SQL : `WHERE`.

| pandas | SQL |
|---|---|
| `==`, `!=`, `>`, `<` | `=`, `<>`, `>`, `<` |
| `&` (avec parenthèses autour de chaque condition) | `AND` |
| `\|` (avec parenthèses) | `OR` |
| `.isin([...])` | `IN (...)` |
| `.isna()` / `.notna()` | `IS NULL` / `IS NOT NULL` |

Comme en SQL, le texte est sensible à la casse : `"Clos"` et `"clos"` sont différents.

In [10]:
# Question : quels clients habitent le Val-d'Oise (95) ?
clients_95 = clients[clients["departement"] == "95"]
clients_95[["nom", "prenom", "ville"]].head()

,nom,prenom,ville
2,Michel,Lea,Argenteuil
6,Dupont,Sophie,Argenteuil
11,Mercier,Moussa,Cergy
13,Diallo,Julie,Cergy
17,David,Yann,Cergy


In [11]:
len(clients_95)   # len() compte les lignes
# Résultat : 31 (comme en SQL)

31

In [12]:
# Question : quels gros sinistres (> 5 000 €) en 2025 ?
# Chaque condition entre parenthèses, reliées par &
gros_2025 = sinistres[
    (sinistres["date_sinistre"].between("2025-01-01", "2025-12-31"))
    & (sinistres["montant_declare"] > 5000)
]
len(gros_2025)
# Résultat : 26

26

In [13]:
# Question : quels sinistres sont des vols ou des cambriolages ?
sinistres[sinistres["type_sinistre"].isin(["Vol", "Cambriolage"])].head()

,id_sinistre,id_contrat,date_sinistre,type_sinistre,montant_declare,montant_rembourse,statut
11,12,19,2022-10-17,Vol,7971.10,0.00,Refusé
15,16,25,2026-04-29,Vol,2033.66,1076.41,Clos
17,18,28,2024-03-25,Cambriolage,7361.26,6492.81,Clos
19,20,30,2020-04-25,Vol,6302.83,0.00,Refusé
28,29,40,2026-02-24,Vol,8649.11,5077.90,Clos


In [14]:
# Question : combien de contrats n'ont pas de prime ?
contrats["prime_annuelle"].isna().sum()
# Résultat : 15

15


1. Combien y a-t-il de clientes (`sexe == "F"`) ?
2. Affiche les contrats **Santé** avec une prime supérieure à 1 500 €.

In [15]:
# Question : quel est le nombre de clientes ?
clients_F = clients[clients["sexe"] == "F"]
len(clients_F)


100

In [16]:
# Question : quels sont les contrats Santé avec prime > 1 500 € ?
sante_cher = contrats[
    (contrats["type_contrat"] == "Santé")
    & (contrats["prime_annuelle"] > 1500)
]
display(sante_cher)
print("Nombre :", len(sante_cher))
# Résultat : 19 contrats


,id_contrat,id_client,type_contrat,date_debut,date_fin,prime_annuelle,statut
0,1,1,Santé,2020-02-11,None,1662.37,Actif
20,21,12,Santé,2023-09-29,None,1755.76,Actif
62,63,39,Santé,2024-01-13,2025-12-02,1532.07,Résilié
90,91,61,Santé,2025-11-23,None,1772.04,Actif
95,96,65,Santé,2023-06-19,None,1744.73,Actif
97,98,66,Santé,2024-01-02,None,1683.07,Actif
112,113,73,Santé,2025-04-27,2026-01-10,1727.97,Résilié
124,125,81,Santé,2025-01-07,2026-03-03,1635.44,Résilié
127,128,83,Santé,2025-12-27,None,1765.12,Actif
130,131,85,Santé,2024-04-24,None,1650.38,Actif


Nombre : 19


## 5. Trier

`df.sort_values("colonne")` trie du plus petit au plus grand ; `ascending=False` inverse l'ordre. Équivalent SQL : `ORDER BY ... DESC`.

Bonne nouvelle : contrairement à SQLite, pandas place les valeurs manquantes **à la fin** du tri.

In [17]:
# Question : quels sont les 5 contrats Santé les plus chers ?
sante = contrats[contrats["type_contrat"] == "Santé"]
sante.sort_values("prime_annuelle", ascending=False).head(5)
# Résultat : contrat n° 146 en tête (1 772,22 €)

,id_contrat,id_client,type_contrat,date_debut,date_fin,prime_annuelle,statut
145,146,94,Santé,2022-03-24,2025-10-26,1772.22,Résilié
90,91,61,Santé,2025-11-23,None,1772.04,Actif
127,128,83,Santé,2025-12-27,None,1765.12,Actif
236,237,144,Santé,2023-10-30,None,1763.92,Actif
20,21,12,Santé,2023-09-29,None,1755.76,Actif



Affiche les 3 contrats **Auto** avec la prime la plus faible.

In [18]:
# Question : quels sont les contrats Auto les moins chers ?
auto = contrats[contrats["type_contrat"] == "Auto"]
auto.sort_values("prime_annuelle").head(3)
# Résultat : contrats n° 11 (376,48 €), 164 (402,81 €), 339 (427,53 €)


,id_contrat,id_client,type_contrat,date_debut,date_fin,prime_annuelle,statut
10,11,7,Auto,2024-11-05,None,376.48,Actif
163,164,103,Auto,2021-12-07,None,402.81,Actif
338,339,200,Auto,2021-05-30,None,427.53,Actif


## 6. Compter et calculer par catégorie

- `df["colonne"].value_counts()` : compte chaque valeur. Équivalent : `GROUP BY` + `COUNT(*)`.
- `df.groupby("colonne")["autre"].agg([...])` : plusieurs calculs par groupe. Équivalent : `GROUP BY` + `COUNT`, `SUM`, `AVG`.
- `.round(2)` : arrondit à 2 décimales.

In [19]:
# Question : combien de contrats y a-t-il par type ?
contrats["type_contrat"].value_counts()
# Résultat : Santé 88, Habitation 88, Prévoyance 83, Auto 80

Santé         88
Habitation    88
Prévoyance    83
Auto          80
Name: type_contrat, dtype: int64

In [20]:
# Question : par type de contrat, nombre de primes, total et moyenne ?
contrats.groupby("type_contrat")["prime_annuelle"].agg(["count", "sum", "mean"]).round(2).sort_values("sum", ascending=False)
# Résultat : Santé en tête (99 084,93 € ; moyenne 1 152,15 €)
# Attention : "count" ignore les primes manquantes, comme COUNT(colonne) en SQL

,count,sum,mean
type_contrat,,,
Santé,86,99084.93,1152.15
Auto,75,57671.13,768.95
Prévoyance,80,44406.20,555.08
Habitation,83,30994.23,373.42



1. Combien de clients par ville ?
2. Pour chaque type de sinistre, le nombre de sinistres et le montant moyen déclaré, du plus coûteux au moins coûteux.

In [21]:
# Question : combien de clients y a-t-il par ville ?
clients["ville"].value_counts()
# Résultat : Évry 23, Argenteuil 20, Paris 19, Nanterre 19…


Évry                    23
Argenteuil              20
Paris                   19
Nanterre                19
Lille                   18
Bobigny                 18
Boulogne-Billancourt    17
Créteil                 17
Lyon                    16
Versailles              11
Saint-Denis             11
Cergy                   11
Name: ville, dtype: int64

In [22]:
# Question : quels sont le nombre et le montant moyen des sinistres par type ?
(sinistres.groupby("type_sinistre")["montant_declare"]
    .agg(["count", "mean"])
    .round(2)
    .sort_values("mean", ascending=False))
# Résultat : Cambriolage en tête (19 sinistres ; 5 465,96 €), Hospitalisation dernier (3 638,61 €)


,count,mean
type_sinistre,,
Cambriolage,19,5465.96
Soins dentaires,33,4650.59
Accident,31,4549.30
Invalidité,29,4514.50
Vol,23,4485.41
Incendie,19,4479.44
Optique,19,4477.63
Dégât des eaux,21,4332.35
Arrêt de travail,27,4234.41


## Bonus — Le taux de remboursement des sinistres clos

On peut calculer avec des colonnes entières, comme dans un tableur.

In [23]:
clos = sinistres[sinistres["statut"] == "Clos"]
taux = 100 * clos["montant_rembourse"].sum() / clos["montant_declare"].sum()
print("Sinistres clos :", len(clos))
print("Taux de remboursement :", round(taux, 1), "%")
# Résultat : 155 sinistres clos ; 76,5 %

Sinistres clos : 155
Taux de remboursement : 76.5 %


## À retenir

| Besoin | SQL | pandas |
|---|---|---|
| Voir les premières lignes | `LIMIT 5` | `df.head()` |
| Choisir des colonnes | `SELECT a, b` | `df[["a", "b"]]` |
| Filtrer | `WHERE a = 'x'` | `df[df["a"] == "x"]` |
| Liste de valeurs | `IN (...)` | `.isin([...])` |
| Valeurs manquantes | `IS NULL` | `.isna()` |
| Trier | `ORDER BY a DESC` | `.sort_values("a", ascending=False)` |
| Compter par groupe | `GROUP BY a` + `COUNT(*)` | `df["a"].value_counts()` |
| Calculs par groupe | `GROUP BY` + `SUM`, `AVG` | `.groupby("a")["b"].agg(["sum", "mean"])` |

In [24]:
# Toujours fermer la connexion à la fin
connexion.close()